# Hurdle-distribution primer: canonical curve → probabilities

Flood depth is zero most of the time and continuous only in the upper tail.
**crc-sdk** encodes that as a *hurdle distribution* — a point mass at zero depth
plus a truncated parametric family above it — and persists the fitted parameters
in canonical Parquet.

This short notebook uses the checked-in Cologne OS-Climate fixture so it runs
offline. It focuses on the fitted result rather than repeating ingestion:

1. **Inspect** the canonical hazard row and its persisted parameters.
2. **Reconstruct** the `HurdleDistribution` from Parquet.
3. **Evaluate and sample** the distribution across useful return periods.

The fixture was produced by the original live SDK smoke test from WRI Aqueduct
riverine flood depth (historical, 1980).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

from crc_sdk import HurdleDistribution
from crc_sdk.connectors import read_hazard_dataset
from crc_sdk.types import CurveParameters

# GitHub's notebook preview only renders static HTML/images (no JS execution),
# so every fig.show() below emits both an interactive widget (for local/Jupyter
# use) and a static PNG fallback (via kaleido) that GitHub's viewer displays.
pio.renderers.default = "jupyterlab+png"

HAZARD_PATH = Path("../fixtures/os_climate/hazard.parquet")
RETURN_PERIODS = np.asarray([2, 5, 10, 25, 50, 100, 250, 500, 1000], dtype=float)
SAMPLES = 10_000
SEED = 42


## 1. Inspect the canonical hazard fixture

The fixture preserves the complete canonical schema and Parquet metadata from
the live ingest. Several H3 cells overlap the same source pixel; one row is
enough to inspect the shared fitted curve.


In [ ]:
hazard = read_hazard_dataset(HAZARD_PATH)
print(f"{hazard.num_rows} canonical rows <- {HAZARD_PATH}")
hazard.to_pandas()[
    [
        "cell_index",
        "pathway",
        "horizon",
        "curve_kind",
        "curve_type",
        "curve_location",
        "curve_scale",
        "curve_atom_probability",
    ]
].head()


## 2. Reconstruct the fitted hurdle

`CurveParameters.to_distribution()` validates the persisted fields and rebuilds
the same framework `HurdleDistribution` that the ingest fitted. The atom models
dry events; the Gumbel tail models positive flood depth.


In [ ]:
row = hazard.slice(0, 1).to_pylist()[0]
fitted = CurveParameters.model_validate(
    {
        key: row[key]
        for key in (
            "curve_kind",
            "curve_type",
            "curve_shape",
            "curve_location",
            "curve_scale",
            "curve_atom_probability",
            "curve_atom_location",
        )
    }
).to_distribution()
assert isinstance(fitted, HurdleDistribution)
print(
    "hurdle: "
    f"atom={fitted.atom_probability}@{fitted.atom_location}, "
    f"{fitted.base.family}(location={fitted.base.location:.4f}, "
    f"scale={fitted.base.scale:.4f}, shape={fitted.base.shape})"
)


## 3. Evaluate return levels and sampling error

The exact PPF gives deterministic return-period depth. Empirical quantiles from
10,000 samples should converge toward it; their residuals quantify Monte Carlo
noise rather than fitting error.


In [ ]:
probabilities = 1.0 - 1.0 / RETURN_PERIODS
exact = fitted.quantiles(probabilities)
samples = fitted.sample(SAMPLES, seed=SEED)
sampled = np.quantile(samples, probabilities)
compare = pd.DataFrame(
    {
        "return_period": RETURN_PERIODS,
        "probability": probabilities,
        "exact_ppf": exact,
        "sampled_quantile": sampled,
        "sampling_residual": sampled - exact,
    }
)
print(
    "sample-vs-curve: "
    f"mae={np.mean(np.abs(compare['sampling_residual'])):.4f}, "
    f"rmse={np.sqrt(np.mean(np.square(compare['sampling_residual']))):.4f}"
)
print(
    "sample atom probability: "
    f"configured={fitted.atom_probability:.4f}, "
    f"empirical={np.mean(samples == fitted.atom_location):.4f}"
)
compare


## 4. Visualize


In [ ]:
fig_fit = go.Figure()
fig_fit.add_trace(
    go.Scatter(
        x=compare["return_period"],
        y=compare["exact_ppf"],
        mode="markers+lines",
        name="exact hurdle PPF",
        marker=dict(size=10, color="#2171B5"),
    )
)
fig_fit.add_trace(
    go.Scatter(
        x=compare["return_period"],
        y=compare["sampled_quantile"],
        mode="markers+lines",
        name="sampled quantile",
        marker=dict(size=8, color="#D94801"),
    )
)
fig_fit.update_layout(
    title="Exact return levels vs sampled quantiles — Cologne",
    xaxis_title="Return period (years)",
    yaxis_title="Flood depth (m)",
    xaxis_type="log",
    margin=dict(l=60, r=40, t=40, b=60),
)
fig_fit.show()


In [ ]:
positive = samples[samples > fitted.atom_location]
fig_hist = go.Figure()
fig_hist.add_trace(
    go.Histogram(
        x=positive,
        nbinsx=40,
        name="tail samples",
        marker_color="#2171B5",
        histnorm="probability density",
    )
)
fig_hist.add_vline(
    x=fitted.atom_location,
    line_dash="dash",
    line_color="#D94801",
    annotation_text=f"atom p={fitted.atom_probability:.2f} at 0 m",
)
fig_hist.update_layout(
    title=f"Monte Carlo samples from the fitted hurdle (n={SAMPLES:,})",
    xaxis_title="Flood depth (m)",
    yaxis_title="Density (positive samples)",
    margin=dict(l=60, r=40, t=40, b=60),
)
fig_hist.show()


## Where this shows up next

- [`asset_portfolio_evaluation.ipynb`](asset_portfolio_evaluation.ipynb) reuses the
  same canonical file to evaluate a small warehouse portfolio.
- [`flood_risk_by_province.ipynb`](flood_risk_by_province.ipynb) applies the same
  hurdle fit over an areal AOI, then joins to admin boundaries.
